In [3]:
import os
import glob
import pandas as pd

# ==========================================
# 1. THE EXCEL-TO-CSV CLEANER
# ==========================================
def convert_and_clean_excel(input_filepath, output_dir):
    """
    Reads a messy .xlsx file, learns the header from the first row of data, 
    removes any repeat headers, and exports directly to a clean .csv.
    """
    # Grab the original filename and create the "_clean.csv" version
    base_name = os.path.basename(input_filepath)
    name_without_ext = os.path.splitext(base_name)[0]
    
    # Strip out any junk Excel adds to the filename (like " - Table 1")
    clean_name = name_without_ext.replace(" - Table 1", "")
    output_filepath = os.path.join(output_dir, f"{clean_name}_clean.csv")

    try:
        # Read the Excel file. header=None prevents Pandas from accidentally 
        # using a blank row at the top of the sheet as the header.
        df = pd.read_excel(input_filepath, header=None)
        
        # 1. Drop completely empty rows
        df.dropna(how='all', inplace=True)
        
        # 2. Grab the first column and make it lowercase string for easy checking
        first_col = df[0].astype(str).str.strip().str.lower()
        
        # 3. Dynamic Header Detection (The first non-empty row dictates the rule)
        master_header_val = first_col.iloc[0]
        
        # 4. Filter the data! We keep a row IF:
        #    - It is the very first row (so we keep the master header)
        #    - OR its first column does NOT match the master header
        mask = (df.index == df.index[0]) | (first_col != master_header_val)
        df_clean = df[mask]
        
        # 5. Export perfectly to CSV (header=False because our data's first row IS the header)
        df_clean.to_csv(output_filepath, index=False, header=False)
        
        print(f"  ✅ Converted & Cleaned: {base_name} -> {os.path.basename(output_filepath)}")
        
    except Exception as e:
        print(f"  ❌ Failed to process '{base_name}'. Error: {e}")


# ==========================================
# 2. THE MASTER DIRECTORY CONTROLLER
# ==========================================
def process_fsae_year_excel(year, root_dir="."):
    """
    Scans the raw_excel_data/{year} folder and routes all .xlsx files 
    through the universal converter.
    """
    year_str = str(year)
    
    # Define your exact input and output paths based on your folder structure
    input_dir = os.path.join(root_dir, "raw_xsxl_data", year_str)
    output_dir = os.path.join(root_dir, "fsae_data", year_str)

    # Safety Check: Does the input directory exist?
    if not os.path.exists(input_dir):
        print(f"❌ Error: Could not find input directory: {input_dir}")
        print("Please ensure your messy Excel files are placed in this folder.")
        return

    # Automatically create the destination folder if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"📂 Processing Excel data for {year_str}...")
    print(f"   Reading from: {input_dir}")
    print(f"   Saving to:    {output_dir}\n")

    # Grab every .xlsx file in that year's folder
    raw_files = glob.glob(os.path.join(input_dir, "*.xlsx"))

    if not raw_files:
        print("   No .xlsx files found in this directory.")
        return

    for filepath in raw_files:
        # We skip any temp files Excel might create (which start with '~$')
        if os.path.basename(filepath).startswith('~$'):
            continue
            
        convert_and_clean_excel(filepath, output_dir)

    print("\n🏁 All Excel files converted and cleaned successfully!")

# ==========================================
# 3. RUN THE SCRIPT
# ==========================================
if __name__ == "__main__":
    # Change this to whatever year you want to process!
    target_year = 2019
    
    # If this script is saved inside the "VroomVroom" folder, root_dir is "."
    process_fsae_year_excel(target_year, root_dir=".")

📂 Processing Excel data for 2019...
   Reading from: ./raw_xsxl_data/2019
   Saving to:    ./fsae_data/2019

  ✅ Converted & Cleaned: fsae_2019_acceleration_result.xlsx -> fsae_2019_acceleration_result_clean.csv
  ✅ Converted & Cleaned: fsae_2019_fuel_efficiency_result.xlsx -> fsae_2019_fuel_efficiency_result_clean.csv
  ✅ Converted & Cleaned: fsae_2019_team_information_result.xlsx -> fsae_2019_team_information_result_clean.csv
  ✅ Converted & Cleaned: fsae_2019_design_result.xlsx -> fsae_2019_design_result_clean.csv
  ✅ Converted & Cleaned: fsae_2019_presentation_result.xlsx -> fsae_2019_presentation_result_clean.csv
  ✅ Converted & Cleaned: fsae_2019_overall_result.xlsx -> fsae_2019_overall_result_clean.csv
  ✅ Converted & Cleaned: fsae_2019_autocross_result.xlsx -> fsae_2019_autocross_result_clean.csv
  ✅ Converted & Cleaned: fsae_2019_endurance_result.xlsx -> fsae_2019_endurance_result_clean.csv
  ✅ Converted & Cleaned: fsae_2019_cost_event_result.xlsx -> fsae_2019_cost_event_resul